In [113]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn import datasets
from sklearn.model_selection import GridSearchCV
import pandas as pd 
import numpy as np

In [114]:
df = pd.read_excel('Working Version - default of credit card clients.xls',header=1)
df = pd.DataFrame(df)
df

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,PAY_AMT5,PAY_AMT6,default payment next month,UTIL_t1,UTIL_t2,UTIL_t3,UTIL_t4,UTIL_t5,UTIL_t6,Average of Util Over 6 Months
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,1,0.195650,0.155100,0.034450,0.000000,0.000000,0.000000,0.064200
1,2,120000,2,2,2,26,-1,2,0,0,...,0,2000,1,0.022350,0.014375,0.022350,0.027267,0.028792,0.027175,0.023718
2,3,90000,2,2,2,34,0,0,0,0,...,1000,5000,0,0.324878,0.155856,0.150656,0.159233,0.166089,0.172767,0.188246
3,4,50000,2,2,1,37,0,0,0,0,...,1069,1000,0,0.939800,0.964660,0.985820,0.566280,0.579180,0.590940,0.771113
4,5,50000,1,2,1,57,-1,0,-1,0,...,689,679,0,0.172340,0.113400,0.716700,0.418800,0.382920,0.382620,0.364463
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000,1,3,1,39,0,0,0,0,...,5000,1000,0,0.858855,0.876432,0.947114,0.400018,0.141986,0.072636,0.549507
29996,29997,150000,1,3,2,43,-1,-1,-1,-1,...,0,0,0,0.011220,0.012187,0.023347,0.059860,0.034600,0.000000,0.023536
29997,29998,30000,1,2,2,37,4,3,2,-1,...,2000,3100,1,0.118833,0.111867,0.091933,0.695933,0.686067,0.645233,0.391644
29998,29999,80000,1,3,1,41,1,-1,0,0,...,52964,1804,1,-0.020563,0.979738,0.953800,0.659675,0.148187,0.611800,0.555440


In [115]:
X = df.loc[:,df.columns != 'default payment next month'].copy()
X = pd.DataFrame(X)
Y = df.loc[:,'default payment next month'].values.copy()


#To be one hot encoded
XC = X.loc[:,['SEX','EDUCATION','MARRIAGE','PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']].values
XC

#To be scaled normallly -> N(0,1)
X = X.loc[:, ~X.columns.isin(['SEX','EDUCATION','MARRIAGE','PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6', 
'Average of Util Over 6 Months', 
'ID'])].values

#Scaling for variables
scaler = StandardScaler()
cate = OneHotEncoder()
X = scaler.fit_transform(X)
XC = cate.fit_transform(XC)

XC = XC.toarray()
X_combined = np.hstack([X, XC])



In [116]:

x_trainv, x_test, y_trainv, y_test = train_test_split(X_combined, Y, test_size=0.1,stratify=Y, random_state = 28)

x_train, x_v, y_train, y_v = train_test_split(x_trainv, y_trainv, test_size=0.222, stratify=y_trainv, random_state = 28)

x_v = torch.tensor(x_v,dtype=torch.float32)
y_v = torch.tensor(y_v,dtype=torch.float32).view(-1, 1)

x_train = torch.tensor(x_train,dtype=torch.float32)
y_train = torch.tensor(y_train,dtype=torch.float32).view(-1, 1)

x_test = torch.tensor(x_test,dtype=torch.float32)
y_test = torch.tensor(y_test,dtype=torch.float32).view(-1, 1)


In [117]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(97, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = Net()

pos_weight = torch.tensor([3.0])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

model.net[-1] = nn.Identity()

optimizer = optim.AdamW(model.parameters(), lr=1e-3)

In [118]:
epochs = 50
patience = 10
best_val_loss = float('inf')
counter = 0

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(x_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()


    model.eval()
    with torch.no_grad():
        val_outputs = model(x_v)
        val_loss = criterion(val_outputs, y_v)

    print(f"Epoch {epoch+1}: Train Loss={loss.item():.4f}, Val Loss={val_loss.item():.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        best_model_state = model.state_dict()
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break

model.load_state_dict(best_model_state)

Epoch 1: Train Loss=1.0145, Val Loss=0.9922
Epoch 2: Train Loss=0.9863, Val Loss=0.9877
Epoch 3: Train Loss=0.9596, Val Loss=0.9828
Epoch 4: Train Loss=0.9437, Val Loss=0.9774
Epoch 5: Train Loss=0.9270, Val Loss=0.9717
Epoch 6: Train Loss=0.9138, Val Loss=0.9656
Epoch 7: Train Loss=0.9054, Val Loss=0.9590
Epoch 8: Train Loss=0.8950, Val Loss=0.9521
Epoch 9: Train Loss=0.8837, Val Loss=0.9448
Epoch 10: Train Loss=0.8794, Val Loss=0.9372
Epoch 11: Train Loss=0.8710, Val Loss=0.9293
Epoch 12: Train Loss=0.8686, Val Loss=0.9213
Epoch 13: Train Loss=0.8619, Val Loss=0.9132
Epoch 14: Train Loss=0.8638, Val Loss=0.9052
Epoch 15: Train Loss=0.8581, Val Loss=0.8973
Epoch 16: Train Loss=0.8543, Val Loss=0.8897
Epoch 17: Train Loss=0.8507, Val Loss=0.8824
Epoch 18: Train Loss=0.8454, Val Loss=0.8755
Epoch 19: Train Loss=0.8456, Val Loss=0.8689
Epoch 20: Train Loss=0.8418, Val Loss=0.8628
Epoch 21: Train Loss=0.8372, Val Loss=0.8572
Epoch 22: Train Loss=0.8353, Val Loss=0.8521
Epoch 23: Train Los

<All keys matched successfully>

In [119]:
model.eval()
with torch.no_grad():
    # ----- VALIDATION -----
    val_outputs = model(x_v)
    val_probs = torch.sigmoid(val_outputs).cpu().numpy()
    y_predval = (val_probs > 0.5).astype(int)
    y_v_np = y_v.cpu().numpy()


    print("----- VALIDATION METRICS -----")
    print(f"Accuracy : {accuracy_score(y_v_np, y_predval):.4f}")
    print(f"Precision: {precision_score(y_v_np, y_predval):.4f}")
    print(f"Recall   : {recall_score(y_v_np, y_predval):.4f}")
    print(f"F1 Score : {f1_score(y_v_np, y_predval):.4f}")
    print(f"ROC AUC  : {roc_auc_score(y_v_np, val_probs):.4f}")

    # ----- TEST -----
    test_outputs = model(x_test)
    test_probs = torch.sigmoid(test_outputs).cpu().numpy()
    y_predtest = (test_probs > 0.5).astype(int)
    y_test_np = y_test.cpu().numpy()

    print("\n----- TEST METRICS -----")
    print(f"Accuracy : {accuracy_score(y_test_np, y_predtest):.4f}")
    print(f"Precision: {precision_score(y_test_np, y_predtest):.4f}")
    print(f"Recall   : {recall_score(y_test_np, y_predtest):.4f}")
    print(f"F1 Score : {f1_score(y_test_np, y_predtest):.4f}")
    print(f"ROC AUC  : {roc_auc_score(y_test_np, test_probs):.4f}")



----- VALIDATION METRICS -----
Accuracy : 0.7759
Precision: 0.4948
Recall   : 0.6086
F1 Score : 0.5458
ROC AUC  : 0.7799

----- TEST METRICS -----
Accuracy : 0.7853
Precision: 0.5123
Recall   : 0.6295
F1 Score : 0.5649
ROC AUC  : 0.7909
